# Kapitel 8: Den Tokenizer Bauen

> "Sprache ist das Kleid des Gedankens." — **Samuel Johnson**, Schriftsteller

---

## Was ist Tokenisierung?

**Tokenisierung** ist der Prozess, Text in Zahlen umzuwandeln, damit neuronale Netze ihn verarbeiten können. Stellen Sie es sich vor wie die Übersetzung von Deutsch in einen Geheimcode, bei dem jedes Wort, Teilwort oder Zeichen eine eindeutige Nummer erhält.

```
"Hallo Welt" → [15496, 995]  (mit GPT-2's Tokenizer)
```

Der Tokenizer funktioniert auch umgekehrt: gegeben Zahlen, erzeugt er Text.

---

## Was Sie lernen werden

- Wie Text durch drei verschiedene Tokenisierungsstrategien zu Zahlen wird
- Warum Tokenisierung auf Zeichenebene einfach, aber ineffizient ist
- Wie Tokenisierung auf Wortebene unbekannte Wörter behandelt und warum die Vokabulargröße wichtig ist
- Der clevere Trick hinter Teilwort-Tokenisierung (BPE), der moderne LLMs antreibt
- Wie man Produktions-Tokenizer wie tiktoken und Hugging Face Transformers verwendet
- Die Eigenheiten und Fallstricke, die Prompts und API-Kosten beeinflussen

---

## Setup

Zuerst installieren wir die benötigten Pakete:

In [ ]:
# Benötigte Pakete installieren
!pip install -q tiktoken transformers torch

## 1. Tokenisierung auf Zeichenebene

Bauen wir den einfachsten Tokenizer: jedes Zeichen als Token behandeln.

**Erinnerung an Python-Klassen:**
- Eine **Klasse** ist ein Bauplan, der Daten und Funktionen zusammenbündelt
- `__init__(self)` wird ausgeführt, wenn Sie ein Objekt erstellen (initialisiert seine Daten)
- `self` bezieht sich auf "dieses spezifische Objekt" (wie "dieses Auto" vs. "Autos allgemein")
- `@property` macht eine Methode zu einem Attribut (keine Klammern nötig)

In [ ]:
class CharTokenizer:
    def __init__(self):
        # Zwei Dictionaries für bidirektionale Suche
        self.char_to_id = {}
        self.id_to_char = {}

    def fit(self, text):
        """Vokabular aus Text aufbauen.
        
        Warum sorted()? Damit das Vokabular deterministisch ist – derselbe Text
        erzeugt immer dieselben IDs. Ohne Sortierung ist die Reihenfolge von
        Python's set zufällig.
        """
        # Eindeutige Zeichen extrahieren - sets behandeln Einzigartigkeit automatisch
        chars = sorted(set(text))
        
        # Jedem Zeichen eine ID zuweisen (beginnend bei 0)
        for i, c in enumerate(chars):
            self.char_to_id[c] = i
            self.id_to_char[i] = c

    def encode(self, text):
        """Text in Liste von Token-IDs umwandeln.
        
        Gibt eine Liste von Ganzzahlen zurück, eine pro Zeichen.
        """
        return [self.char_to_id[c] for c in text]

    def decode(self, ids):
        """Liste von Token-IDs zurück in Text umwandeln.
        
        Verwendet str.join(), um Zeichen ohne Leerzeichen zu verketten
        (im Gegensatz zur Wort-Tokenisierung, die Leerzeichen benötigt).
        """
        return "".join(self.id_to_char[i] for i in ids)

    @property
    def vocab_size(self):
        """Wie viele eindeutige Zeichen wir kennen."""
        return len(self.char_to_id)

### Probieren Sie es aus: Vollständiges Beispiel

In [ ]:
# Tokenizer erstellen und trainieren
tokenizer = CharTokenizer()
tokenizer.fit("Hallo, Welt!")

print(f"Vokabulargröße: {tokenizer.vocab_size}")
print(f"Vokabular: {tokenizer.char_to_id}")

# Kodieren
text = "Hallo"
ids = tokenizer.encode(text)
print(f"\n'{text}' → {ids}")

# Dekodieren - Rundweg sollte verlustfrei sein!
decoded = tokenizer.decode(ids)
print(f"{ids} → '{decoded}'")
print(f"Perfekter Rundweg? {text == decoded}")

### Der Kompromiss zwischen Vokabulargröße und Sequenzlänge

In [ ]:
sentence = "Tokenisierung ist der erste Schritt in jedem Sprachmodell."

# Zeichenebene
char_tok = CharTokenizer()
char_tok.fit(sentence)
char_ids = char_tok.encode(sentence)

print(f"Originaltext: {len(sentence)} Zeichen")
print(f"Vokabulargröße: {char_tok.vocab_size}")
print(f"Sequenzlänge: {len(char_ids)}")
print(f"Erste 20 Tokens: {char_ids[:20]}")

## 2. Tokenisierung auf Wortebene

Jetzt bauen wir einen Tokenizer auf Wortebene, der unbekannte Wörter mit speziellen Tokens behandelt.

**Spezielle Tokens:** Reservierte IDs mit spezifischen Bedeutungen:
- `<PAD>` (ID 0): Padding – füllt Sequenzen auf gleiche Länge für Batch-Verarbeitung
- `<UNK>` (ID 1): Unbekannt – repräsentiert Wörter, die nicht in unserem Vokabular sind
- `<BOS>` (ID 2): Sequenzanfang – markiert, wo Text beginnt
- `<EOS>` (ID 3): Sequenzende – markiert, wo Text endet

Warum brauchen wir diese? Ohne `<UNK>` würde unser Tokenizer bei neuen Wörtern abstürzen. Ohne `<PAD>` könnten wir nicht mehrere Sätze gleichzeitig verarbeiten (sie hätten unterschiedliche Längen).

In [ ]:
from collections import Counter

class WordTokenizer:
    def __init__(self, max_vocab_size=10000):
        """
        max_vocab_size: Maximale Vokabulargröße (einschließlich spezieller Tokens)
        
        Warum 10.000? Es ist ein Kompromiss:
        - Zu klein (1.000): Zu viele unbekannte Wörter
        - Zu groß (100.000): Riesige Embedding-Tabelle, langsames Training
        - 10.000-50.000: Optimaler Bereich fürs Lernen
        """
        self.max_vocab_size = max_vocab_size
        self.word_to_id = {}
        self.id_to_word = {}

    def fit(self, text):
        """Vokabular aus häufigsten Wörtern aufbauen."""
        # Einfach beginnen: an Leerzeichen trennen und in Kleinbuchstaben umwandeln
        words = text.lower().split()
        
        # Worthäufigkeiten zählen - warum? Häufige Wörter bekommen eigene IDs,
        # seltene Wörter werden zu <UNK>. Dies minimiert Unbekannte in der Praxis.
        counts = Counter(words)
        
        # IDs 0-3 für spezielle Tokens reservieren
        self.word_to_id = {
            "<PAD>": 0,
            "<UNK>": 1,
            "<BOS>": 2,
            "<EOS>": 3
        }
        self.id_to_word = {v: k for k, v in self.word_to_id.items()}
        
        # Häufigste Wörter hinzufügen (max_vocab_size-Limit beachten)
        # IDs bei 4 starten, da 0-3 für spezielle Tokens reserviert sind
        for i, (word, count) in enumerate(counts.most_common(self.max_vocab_size - 4), start=4):
            self.word_to_id[word] = i
            self.id_to_word[i] = word

    def encode(self, text, add_special_tokens=False):
        """Text in Token-IDs umwandeln.
        
        add_special_tokens: Wenn True, <BOS> am Anfang und <EOS> am Ende hinzufügen
        """
        words = text.lower().split()
        
        # Jedes Wort nachschlagen, auf <UNK> (ID=1) zurückfallen, wenn nicht gefunden
        # .get(word, 1) gibt 1 zurück, wenn Wort nicht in unserem Vokabular ist
        ids = [self.word_to_id.get(w, 1) for w in words]
        
        if add_special_tokens:
            ids = [2] + ids + [3]  # [<BOS>] + Text + [<EOS>]
        
        return ids

    def decode(self, ids, skip_special_tokens=True):
        """Token-IDs zurück in Text umwandeln.
        
        skip_special_tokens: Wenn True, <PAD>, <BOS> etc. nicht ausgeben
        Warum? Sie wollen keine Ausgabe wie: "<BOS> Hallo Welt <EOS>"
        """
        words = []
        for i in ids:
            word = self.id_to_word.get(i, "<UNK>")
            # Spezielle Tokens in Ausgabe überspringen, wenn gewünscht
            if skip_special_tokens and word in ["<PAD>", "<BOS>", "<EOS>"]:
                continue
            words.append(word)
        
        # Mit Leerzeichen verbinden (im Gegensatz zum Char-Tokenizer, der "".join verwendet)
        return " ".join(words)

    @property
    def vocab_size(self):
        """Aktuelle Vokabulargröße (Anzahl eindeutiger Tokens)."""
        return len(self.word_to_id)

### Probieren Sie es aus: Vollständiges Beispiel mit unbekannten Wörtern

In [ ]:
# Tokenizer mit kleinem Vokabular erstellen, um Unbekannte zu erzwingen
tokenizer = WordTokenizer(max_vocab_size=10)

# Mit begrenztem Text trainieren
training_text = """
Die Katze saß auf der Matte.
Die Katze war auf der Matte.
Der Hund saß auf der Matte.
"""
tokenizer.fit(training_text)

print(f"Vokabular: {tokenizer.word_to_id}")

# Satz mit bekannten Wörtern kodieren
text1 = "die katze saß"
ids1 = tokenizer.encode(text1)
print(f"\n'{text1}' → {ids1}")
print(f"Dekodiert: '{tokenizer.decode(ids1)}'")

# Mit unbekanntem Wort kodieren
text2 = "der elefant saß"  # "elefant" nicht im Vokabular!
ids2 = tokenizer.encode(text2)
print(f"\n'{text2}' → {ids2}")
print(f"Dekodiert: '{tokenizer.decode(ids2)}'")

# Mit speziellen Tokens versuchen
ids3 = tokenizer.encode("die katze", add_special_tokens=True)
print(f"\nMit speziellen Tokens: {ids3}")
print(f"Dekodiert (mit Speziellen): '{tokenizer.decode(ids3, skip_special_tokens=False)}'")
print(f"Dekodiert (ohne Spezielle): '{tokenizer.decode(ids3, skip_special_tokens=True)}'")

### Vergleichsübung: Sehen Sie den Kompromiss

In [ ]:
text = "Der schnelle braune Fuchs springt über den faulen Hund"

# Zeichen-Tokenizer
char_tok = CharTokenizer()
char_tok.fit(text)
char_ids = char_tok.encode(text)

# Wort-Tokenizer
word_tok = WordTokenizer(max_vocab_size=20)
word_tok.fit(text)
word_ids = word_tok.encode(text)

print("ZEICHEN-TOKENIZER:")
print(f"  Vokabulargröße: {char_tok.vocab_size}")
print(f"  Sequenzlänge: {len(char_ids)}")
print(f"  Tokens: {char_ids[:20]}...")

print("\nWORT-TOKENIZER:")
print(f"  Vokabulargröße: {word_tok.vocab_size}")
print(f"  Sequenzlänge: {len(word_ids)}")
print(f"  Tokens: {word_ids}")

## Wie Produktions-Tokenizer funktionieren: BPE (Byte-Pair-Kodierung)

Moderne LLMs verwenden **Teilwort-Tokenisierung** – ein cleverer Mittelweg zwischen Zeichen und Wörtern:

**Das Problem:**
- Zeichen-Tokenizer: Zu viele Tokens pro Text (langsam, teuer)
- Wort-Tokenizer: Können keine neuen Wörter verarbeiten ("ChatGPT" → `<UNK>`)

**Die Lösung: BPE (Byte-Pair-Kodierung)**

BPE lernt Teilwörter automatisch, indem wiederholt die häufigsten Zeichenpaare zusammengeführt werden:

```
Schritt 1: Beginne mit Zeichen: ["l", "o", "w", "e", "r"]
Schritt 2: Häufigstes Paar ist ("l", "o") → zu "lo" zusammenführen
Schritt 3: Häufigstes Paar ist ("lo", "w") → zu "low" zusammenführen
Schritt 4: Fortsetzen, bis Vokabulargröße erreicht...
```

**Ergebnis:** Häufige Wörter werden einzelne Tokens, seltene Wörter werden in bekannte Teile aufgeteilt:
- "lower" → ["low", "er"] ✓ (häufig, effizient)
- "lowest" → ["low", "est"] ✓ (zusammengesetztes Wort behandelt!)
- "ChatGPT" → ["Chat", "G", "PT"] ✓ (kein `<UNK>` nötig!)

**Wichtige Einsicht:** BPE erzeugt nie `<UNK>`, weil jede Zeichenfolge in bekannte Teile zerlegt werden kann!

---

## 3. Produktions-Tokenizer: tiktoken

Jetzt verwenden wir OpenAIs tiktoken-Bibliothek für GPT-4's Tokenizer.

## Trainieren Sie Ihren eigenen BPE-Tokenizer

Jetzt sehen wir BPE-Lernen in Aktion! Wir trainieren unseren eigenen Tokenizer auf Shakespeare mit der HuggingFace `tokenizers`-Bibliothek.

**Warum das tun?**
- Muster automatisch entstehen sehen ("ing", "the", "tion")
- Intuition dafür entwickeln, wie Tokenizer funktionieren
- Domain-Mismatch verstehen (Shakespeare-Tokenizer vs. moderner Text)

Zuerst installieren wir die tokenizers-Bibliothek:

In [ ]:
# Die tokenizers-Bibliothek installieren (getrennt von transformers!)
!pip install -q tokenizers

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

print("Imports erfolgreich!")

### TinyShakespeare herunterladen

Wir verwenden denselben Datensatz, mit dem Sie Ihr vollständiges LLM in Kapitel 12 trainieren werden:

In [ ]:
import urllib.request

# TinyShakespeare herunterladen (derselbe Datensatz, den wir in Kapitel 12 verwenden werden!)
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "shakespeare.txt")

# Lesen und inspizieren
with open("shakespeare.txt", "r") as f:
    text = f.read()

print(f"Datensatzgröße: {len(text):,} Zeichen")
print(f"\nErste 200 Zeichen:")
print(text[:200])

### Den BPE-Tokenizer trainieren

Jetzt kommt der spannende Teil - sehen Sie zu, wie BPE Muster aus Shakespeare lernt!

In [ ]:
# Einen leeren BPE-Tokenizer erstellen
bpe_tokenizer = Tokenizer(BPE(unk_token="<UNK>"))

# An Leerzeichen trennen vor BPE-Zusammenführung
bpe_tokenizer.pre_tokenizer = Whitespace()

# Den Trainer konfigurieren
trainer = BpeTrainer(
    vocab_size=1000,           # Klein zum Lernen (Produktion verwendet 30.000+)
    min_frequency=2,           # Token muss mindestens zweimal erscheinen
    special_tokens=["<PAD>", "<UNK>", "<BOS>", "<EOS>"]
)

# Auf Shakespeare trainieren (dauert nur Sekunden!)
bpe_tokenizer.train(files=["shakespeare.txt"], trainer=trainer)

print(f"Vokabulargröße: {bpe_tokenizer.get_vocab_size()}")

### Testen Sie Ihren trainierten Tokenizer

In [ ]:
# Mit Shakespeare-artigem Text testen
test_text = "Sein oder nicht sein, das ist hier die Frage."
output = bpe_tokenizer.encode(test_text)

print(f"Text: '{test_text}'")
print(f"Tokens: {output.tokens}")
print(f"Token-IDs: {output.ids}")
print(f"Anzahl der Tokens: {len(output.ids)}")

# Rundweg verifizieren
decoded = bpe_tokenizer.decode(output.ids)
print(f"\nDekodiert: '{decoded}'")
print(f"Perfekter Rundweg: {test_text == decoded}")

### Domain-Mismatch: Moderner Text auf Shakespeare-Tokenizer

Was passiert, wenn wir modernen technischen Text versuchen, den Shakespeare nie schrieb?

In [ ]:
# Modernen Text versuchen, den Shakespeare nie schrieb
modern_text = "ChatGPT generiert erstaunliche neuronale Netzwerk-Antworten"
output2 = bpe_tokenizer.encode(modern_text)

print(f"Text: '{modern_text}'")
print(f"Tokens: {output2.tokens}")
print(f"Anzahl der Tokens: {len(output2.ids)}")

# Beachten Sie: "ing" ist ein Token (Shakespeare verwendete es!), aber "ChatGPT" wird in Teile aufgeteilt
print("\nWichtige Einsicht:")
print("- 'ing' ist ein einzelnes Token (Shakespeare verwendete oft '-ing'-Wörter)")
print("- 'ChatGPT' wird in Teile aufgeteilt (Shakespeare schrieb nie über KI!)")
print("- Das ist Domain-Mismatch in Aktion")

---

## Produktions-Tokenizer: tiktoken

Jetzt haben Sie gesehen, wie BPE von Grund auf lernt. Sie verstehen, was in diesen Produktions-Tokenizern steckt. Wenn tiktoken "ChatGPT" in Teile aufteilt, wissen Sie warum!

Verwenden wir die Produktionswerkzeuge, die Sie tatsächlich in Ihren Projekten verwenden werden.

In [ ]:
import tiktoken

# GPT-4's Tokenizer-Encoding laden
enc = tiktoken.get_encoding("cl100k_base")  # GPT-4, GPT-3.5-turbo

# Text kodieren
text = "Hallo, Welt! Wie geht es dir?"
tokens = enc.encode(text)

print(f"Text: '{text}'")
print(f"Tokens: {tokens}")
print(f"Anzahl der Tokens: {len(tokens)}")

# Zurück dekodieren
decoded = enc.decode(tokens)
print(f"Dekodiert: '{decoded}'")
print(f"Perfekter Rundweg: {text == decoded}")

### Die tatsächlichen Token-Strings sehen

In [ ]:
# Jedes Token einzeln dekodieren, um zu sehen, was es repräsentiert
token_strings = [enc.decode([t]) for t in tokens]

print(f"Token-Aufschlüsselung:")
for token_id, token_str in zip(tokens, token_strings):
    print(f"  {token_id:5d} → '{token_str}'")

### Token-Zählung für API-Budgetierung

In [ ]:
prompts = [
    "Schreibe ein Haiku über Programmierung.",
    "Erkläre Quantencomputing in einfachen Worten für ein 10-jähriges Kind.",
    "Generiere einen 500-Wörter-Essay über Klimawandel."
]

enc = tiktoken.get_encoding("cl100k_base")

for prompt in prompts:
    tokens = enc.encode(prompt)
    # Beispielpreisgestaltung (aktuelle Tarife auf openai.com/api/pricing prüfen)
    cost = len(tokens) * 0.00001  # $0.01 pro 1K Eingabe-Tokens
    
    print(f"Prompt: '{prompt}'")
    print(f"  Tokens: {len(tokens)}")
    print(f"  Kosten (Eingabe): ~${cost:.5f}\n")

## 4. Produktions-Tokenizer: Hugging Face Transformers

**Hugging Face** ist ein Unternehmen/eine Bibliothek, die vortrainierte Modelle und Tokenizer bereitstellt. `AutoTokenizer` lädt automatisch den richtigen Tokenizer für jedes Modell.

Der Hugging Face Tokenizer gibt ein Dictionary zurück mit:
- `input_ids`: Die Token-IDs (worauf es uns am meisten ankommt)
- `attention_mask`: 1en für echte Tokens, 0en für Padding (sagt dem Modell, was es ignorieren soll)

In [ ]:
from transformers import AutoTokenizer

# GPT-2-Tokenizer laden (Open-Source)
tokenizer = AutoTokenizer.from_pretrained("gpt2")

text = "Hallo, Welt! Wie geht es dir?"

# Kodieren - gibt Dictionary mit Token-IDs und Attention-Mask zurück
encoded = tokenizer(text, return_tensors="pt")  # "pt" = PyTorch-Tensoren

print(f"Text: '{text}'")
print(f"Token-IDs: {encoded['input_ids']}")
print(f"Attention-Mask: {encoded['attention_mask']}")

# Dekodieren
decoded = tokenizer.decode(encoded['input_ids'][0])
print(f"Dekodiert: '{decoded}'")

### Token-Aufschlüsselung

In [ ]:
tokens = encoded['input_ids'][0].tolist()
token_strings = [tokenizer.decode([t]) for t in tokens]

print(f"Token-Aufschlüsselung:")
for tid, tstr in zip(tokens, token_strings):
    print(f"  {tid:5d} → '{tstr}'")

## 5. Eigenheiten der Tokenisierung

### Eigenheit #1: Führende Leerzeichen ändern alles

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

# Mit und ohne führendes Leerzeichen vergleichen
texts = ["Hallo", " Hallo", "Welt", " Welt"]

for text in texts:
    tokens = enc.encode(text)
    print(f"'{text}' → {tokens} ({len(tokens)} Token{'s' if len(tokens) > 1 else ''})")

### Eigenheit #2: Zahlen werden nach Ziffern aufgeteilt

In [ ]:
numbers = ["10", "100", "1000", "10000", "42", "2024"]

enc = tiktoken.get_encoding("cl100k_base")

for num in numbers:
    tokens = enc.encode(num)
    token_strs = [enc.decode([t]) for t in tokens]
    print(f"'{num}' → {tokens} = {token_strs}")

### Eigenheit #3: Emoji und Sonderzeichen

In [ ]:
emojis = ["😀", "🚀", "👍", "Hallo 😀 Welt", "🔥🔥🔥"]

enc = tiktoken.get_encoding("cl100k_base")

for text in emojis:
    tokens = enc.encode(text)
    token_strs = [enc.decode([t]) for t in tokens]
    print(f"'{text}' → {len(tokens)} Tokens: {token_strs}")

## 6. Praktische Übung: Tokenisieren Sie Ihren Datensatz

Verbinden Sie den Datensatz aus Kapitel 7 mit Tokenisierung (Sie benötigen Ihre chapter7_output.jsonl-Datei dafür):

In [ ]:
import json
import tiktoken

# Beispiel-Datensatz (ersetzen Sie durch Ihre Ausgabe aus Kapitel 7)
example_dataset = [
    {"text": "KI-Systeme lernen aus Beispielen", "split": "train"},
    {"text": "Neuronale Netze benötigen viele Daten", "split": "train"},
    {"text": "Deep Learning verwendet mehrere Schichten", "split": "val"}
]

# Jedes Beispiel tokenisieren
enc = tiktoken.get_encoding("cl100k_base")

for record in example_dataset:
    text = record["text"]
    tokens = enc.encode(text)
    record["token_ids"] = tokens
    record["token_count"] = len(tokens)

print(f"{len(example_dataset)} Beispiele tokenisiert")

# Statistiken berechnen
token_counts = [r["token_count"] for r in example_dataset]
avg_tokens = sum(token_counts) / len(token_counts)
max_tokens = max(token_counts)
min_tokens = min(token_counts)

print(f"\nStatistiken:")
print(f"  Durchschnittliche Tokens pro Beispiel: {avg_tokens:.1f}")
print(f"  Max. Tokens: {max_tokens}")
print(f"  Min. Tokens: {min_tokens}")

# Erstes Beispiel anzeigen
print(f"\nErstes Beispiel:")
print(json.dumps(example_dataset[0], indent=2, ensure_ascii=False))

## Kapitel-Zusammenfassung

**Was wir gebaut haben:**

1. **Zeichen-Tokenizer:** Einfach, aber ineffizient (winziges Vokabular ~100, lange Sequenzen)
2. **Wort-Tokenizer:** Effiziente Sequenzen, aber riesiges Vokabular und Probleme mit unbekannten Wörtern
3. **BPE-Tokenizer:** Unseren eigenen auf Shakespeare trainiert, Muster automatisch entstehen sehen!
4. **Produktionswerkzeuge:** tiktoken und Hugging Face für reale Tokenisierung verwendet

**Was wir gelernt haben:**

- Tokenisierung ist reversibel (verlustfreier Rundweg)
- Der Kompromiss zwischen Vokabulargröße und Sequenzlänge ist fundamental
- Spezielle Tokens dienen spezifischen Zwecken (BOS/EOS/PAD/UNK)
- BPE lernt Teilwörter automatisch durch Zusammenführen häufiger Paare
- Domain-Mismatch ist wichtig: Shakespeare-Tokenizer hat Schwierigkeiten mit modernen Tech-Wörtern
- Tokenisierung hat Eigenheiten (führende Leerzeichen, Zahlenaufteilung, Emoji)

**Nächstes:** Kapitel 9 wird diese Token-IDs in Einbettungsvektoren umwandeln!